# persona-hexaco results: personality profiles, their stability under prompting, and what the states carry

Interprets and graphs what `persona-hexaco.ipynb` left in `~/persona-hexaco-output`. It loads no model and needs
no card: CPU torch is enough, so it also runs on a copy of the outputs off the rig (`PERSONA_HEXACO_OUTPUT`
points at the copy, `GPU_TELEMETRY` at a copy of the telemetry).

| Section | What | Question |
|---|---|---|
| 1 | **Profiles** | Nine polar diagrams per model: each factor's mean and spread on the 1-5 scale, one diagram per prompt |
| 2 | Answers and capture cost | How many items were rated, by whom, how long replies were, how long capture took |
| 3 | **Stable and unstable traits** | Which factors hold still when the prompt changes, per model? |
| 4 | **Shifts from the prompts** | What do DAN and INFO do to each factor, overall and per model? |
| 5 | Factor identification | Do states and RP-LSH digests say which factor an item is about? |
| 6 | Personality direction | Per factor, a direction from the train half's answers, tested on the eval half |
| 7 | **Direction and magnitude** | Are the factors' directions separable from each other, and is magnitude separable from direction? |
| 8 | Prompts as directions | Does a digest identify the differential prompt? |
| 9 | The join | Does the shift in LSH space match the shift in the scores? |
| 10 | Layer sweep | Where in depth do 5, 6 and 8 peak? |
| 11 | The cards during the run | Time per phase, power, temperature, kernel messages |
| 12 | What to read off | The questions each table answers |

**Design, in short.** 136 items (IPIP-NEO-120 + the 16 HEXACO-100 Honesty-Humility items), nine prompts
(DAN0-2 x INFO0-2, baseline D1I2 "answer this question"), no persona instruction. Each factor's items are cut
in half by the experiment's seed: directions and means come from the **train** half only, and every accuracy
and correlation marked *held-out* is on the **eval** half. Sections 1 to 4 are black-box (ratings only) and use
all items; 5 to 10 are white-box.

**Two vectors per answer.** `prompt` is the state at the last prompt token, before any reply token; `response`
is the mean over the reply's tokens (persona-lsh's vector). Where the reply is one digit, `response` mostly
reads that digit, so a `response` result is about the words and a `prompt` result is about the state before them.

**Sources.**
- IPIP-NEO-120: Johnson, J. A. (2014), Journal of Research in Personality, 51, 78-89; public-domain items from
  [ipip.ori.org](https://ipip.ori.org); texts and keying as packaged by
  [five-factor-e](https://github.com/NeuroQuestAi/five-factor-e) 1.13.1.
- Honesty-Humility: HEXACO-PI-R 100-item self-report form and
  [scoring key](https://hexaco.org/downloads/ScoringKeys_100.pdf),
  [hexaco.org/hexaco-inventory](https://hexaco.org/hexaco-inventory) (Lee & Ashton; Lee & Ashton (2018),
  Assessment, 25, 543-556). Free for non-profit academic research only: the item text stays in the output
  directory and out of this repo, and tables here show item ids, not text.
- Code: `personality/hexaco.py`; classification by nearest mean and RP-LSH as in this repo's `demo/` and `lsh/`.
  The notebook follows the harness's `persona-lsh-results.ipynb` in layout and chart style.

In [ ]:
import json
import os
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

PERSONA_LSH = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lsh" / "rp_backend.py").is_file())
sys.path.insert(0, str(PERSONA_LSH))
import personality.hexaco as hx
from lsh import RandomProjectionBackend

OUT_ROOT = Path(os.environ.get("PERSONA_HEXACO_OUTPUT", Path.home() / "persona-hexaco-output"))
EXPERIMENT = "latest"            # a directory name under OUT_ROOT, or "latest": the newest experiment.json
ONLY = []                        # run names to limit the analysis to; empty = every finished run
TELEMETRY_ROOT = Path(os.environ.get("GPU_TELEMETRY", Path.home() / "gpu-telemetry"))
TELEMETRY_DIR = None             # None: the newest persona-hexaco_* with a capture step; or a directory

SPREAD = "sd"                    # the band in the polar diagrams: "sd" = mean +/- 1 standard deviation over the
                                 # factor's items; "variance" = mean +/- the variance itself (in squared points)
KINDS = ["prompt", "response"]   # vectors to analyze; thinking runs also have "answer"
LAYER = None                     # None: 20/28 of depth, persona-lsh's default single layer; or a layer number
RP_BITS = [256, 1024]
RP_SEED = 42
CV_FOLDS, CV_SEED = 3, 42
BOOT, BOOT_SEED = 2000, 42       # bootstrap resamples (over items) for the 95% intervals
SAVE_FIGS = True                 # PNGs into <experiment>/results/figures

pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_rows", 200)
pct = lambda v: "" if v is None or pd.isna(v) else f"{v:.1%}"
num = lambda v: "" if v is None or pd.isna(v) else f"{v:+.2f}"
two = lambda v: "" if v is None or pd.isna(v) else f"{v:.2f}"


def show(df, formats=None, index=False):
    formats = formats or {}
    print(df.to_string(index=index, na_rep="", formatters={c: f for c, f in formats.items() if c in df}))

### Chart style

The palette of `persona-lsh-results.ipynb`: categorical slots in a fixed order, a one-hue blue ramp for
magnitudes, and blue-gray-red for signed values. Identity never rests on color alone: series also differ in
marker or line style, and every chart has a legend or direct labels and a printed table.

In [ ]:
SURFACE, INK, INK2, MUTED, GRID, BASELINE = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]
BLUE, ORANGE = SLOTS[0], SLOTS[1]
SERIOUS = "#ec835a"   # status color: kernel errors, always with a text label
SEQ = LinearSegmentedColormap.from_list("seq", ["#f0efec", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
DIV = LinearSegmentedColormap.from_list("div", ["#104281", "#5598e7", "#f0efec", "#ec8a89", "#a52a29"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": BASELINE, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "lines.linewidth": 2, "figure.dpi": 110})


def save(fig, name):
    if SAVE_FIGS:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=150)


def heat(ax, table, cmap, vmin, vmax, fmt, marks=None):
    # An annotated heatmap of a DataFrame; marks: a same-shaped boolean frame whose True cells get a "*".
    values = table.to_numpy(dtype=float)
    ax.imshow(np.ma.masked_invalid(values), cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    span = max(abs(vmin), abs(vmax))
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            v = values[i, j]
            if np.isfinite(v):
                dark = (abs(v) if vmin < 0 else v - vmin) > 0.6 * (span if vmin < 0 else vmax - vmin)
                star = "*" if marks is not None and bool(marks.iloc[i, j]) else ""
                ax.text(j, i, fmt(v) + star, ha="center", va="center", fontsize=8, color="white" if dark else INK)
    ax.set_xticks(range(values.shape[1]), table.columns)
    ax.set_yticks(range(values.shape[0]), table.index)
    ax.grid(False)
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)


def grid_of(n, per_row=4, width=3.1, height=2.9, **kwargs):
    # Small multiples: (figure, flat list of n axes); spare axes are hidden.
    rows = -(-n // per_row)
    fig, axes = plt.subplots(rows, min(n, per_row), figsize=(width * min(n, per_row), height * rows), squeeze=False, **kwargs)
    flat = list(axes.flat)
    for ax in flat[n:]:
        ax.set_visible(False)
    return fig, flat[:n]

## 0. Load the experiment

A run is analyzed when all its conditions are captured and rated. `RAW[run]` holds the ratings as
`[conditions, items]` (NaN where the model gave none), `KEYED[run]` the same in the factor's direction
(reverse-keyed items flipped, so 5 is always the most of the factor).

In [ ]:
if EXPERIMENT == "latest":
    found = sorted(OUT_ROOT.glob("*/experiment.json"), key=lambda p: p.stat().st_mtime)
    assert found, f"no experiment in {OUT_ROOT}: run persona-hexaco.ipynb first"
    OUT_DIR = found[-1].parent
else:
    OUT_DIR = OUT_ROOT / EXPERIMENT
DESIGN = json.loads((OUT_DIR / "experiment.json").read_text())
ITEMS = hx.items_from_records(json.loads((OUT_DIR / "items.json").read_text()))
SPLIT = json.loads((OUT_DIR / "selection.json").read_text())["factors"]
CONDITIONS = [c for c in hx.conditions() if c in DESIGN["prompts"]]
BASE = CONDITIONS.index(DESIGN["baseline"])
OTHERS = [c for c in CONDITIONS if c != DESIGN["baseline"]]
FACTORS = list(SPLIT)
RESULTS = OUT_DIR / "results"
RESULTS.mkdir(exist_ok=True)
FIG_DIR = RESULTS / "figures"

INDEX = {item.id: k for k, item in enumerate(ITEMS)}
TRAIN = {f: np.array([INDEX[i] for i in SPLIT[f]["train"]]) for f in FACTORS}
EVAL = {f: np.array([INDEX[i] for i in SPLIT[f]["eval"]]) for f in FACTORS}
ALL_TRAIN, ALL_EVAL = np.concatenate(list(TRAIN.values())), np.concatenate(list(EVAL.values()))
REVERSE = np.array([item.reverse for item in ITEMS])
FACTOR_OF = np.array([item.factor for item in ITEMS])
ITEMS_OF = {f: np.flatnonzero(FACTOR_OF == f) for f in FACTORS}
LEVELS = {c: hx.levels(c) for c in CONDITIONS}
COND_AT = {LEVELS[c]: CONDITIONS.index(c) for c in CONDITIONS}


def finished(name):
    path = OUT_DIR / name / "ratings.json"
    return (path.exists() and all(c in json.loads(path.read_text()) for c in CONDITIONS)
            and all((OUT_DIR / name / f"states_{c}.pt").exists() for c in CONDITIONS))


RUN_NAMES = [p.name for p in sorted(OUT_DIR.iterdir()) if p.is_dir() and p.name != "results" and finished(p.name)
             and (not ONLY or p.name in ONLY)]
assert RUN_NAMES, f"no finished run in {OUT_DIR}"
RATINGS = {n: json.loads((OUT_DIR / n / "ratings.json").read_text()) for n in RUN_NAMES}
RUN_INFO = {n: json.loads((OUT_DIR / n / "run_info.json").read_text()) for n in RUN_NAMES}
RESPONSES = {n: json.loads((OUT_DIR / n / "responses.json").read_text()) for n in RUN_NAMES}
RAW, KEYED = {}, {}
for name in RUN_NAMES:
    for c in CONDITIONS:
        assert [r["item"] for r in RATINGS[name][c]] == [i.id for i in ITEMS], f"{name}/{c}: items out of order"
    RAW[name] = np.array([[r["rating"] if isinstance(r["rating"], int) else np.nan for r in RATINGS[name][c]]
                          for c in CONDITIONS], dtype=float)
    KEYED[name] = np.where(REVERSE, 6 - RAW[name], RAW[name])

print("experiment:", OUT_DIR.name, "| created", DESIGN.get("created"), "| seed", DESIGN["seed"])
print("runs:", RUN_NAMES)
print("items per factor (train + eval):", {f: f"{len(TRAIN[f])} + {len(EVAL[f])}" for f in FACTORS})
print("factors:", {f: hx.FACTORS[f] for f in FACTORS})
for c in CONDITIONS:
    print(f"  {c}{' (baseline)' if c == DESIGN['baseline'] else ''}: {DESIGN['prompts'][c]}")

## 1. Profiles: nine polar diagrams per model

One figure per model, one diagram per prompt (rows DAN0-2, columns INFO0-2). Each spoke is a factor; the radius
is the 1-5 scale in the factor's direction (5 = the most of it), with the neutral 3 drawn as a darker ring.

- **Line and dots:** the factor's mean keyed rating over its rated items (24 per Big Five factor, 16 for H).
- **Band:** the spread over those items, mean +/- 1 standard deviation (`SPREAD = "sd"`). Set
  `SPREAD = "variance"` for a band of mean +/- the variance instead; the table prints both. A wide band means
  the model rates the factor's items unevenly (some high, some low), a narrow one that it rates them alike.
- **Dashed gray:** the baseline prompt's profile (D1I2), repeated in every diagram so a shift shows at a glance.

A model that answers 3 to everything draws a small regular hexagon on the dark ring in all nine diagrams.

In [ ]:
MEAN, SD, RATED = {}, {}, {}
rows = []
for name in RUN_NAMES:
    MEAN[name], SD[name], RATED[name] = (np.full((len(CONDITIONS), len(FACTORS)), np.nan) for _ in range(3))
    for ci, c in enumerate(CONDITIONS):
        for fi, f in enumerate(FACTORS):
            k = KEYED[name][ci][ITEMS_OF[f]]
            k = k[np.isfinite(k)]
            RATED[name][ci, fi] = len(k)
            if len(k):
                MEAN[name][ci, fi], SD[name][ci, fi] = k.mean(), k.std()
            rows.append({"run": name, "prompt": c, "factor": f, "rated": len(k), "of": len(ITEMS_OF[f]),
                         "mean": MEAN[name][ci, fi], "sd": SD[name][ci, fi], "variance": SD[name][ci, fi] ** 2})
PROFILE = pd.DataFrame(rows)
PROFILE.to_csv(RESULTS / "profile.csv", index=False)
wide = PROFILE.assign(cell=[f"{m:.2f} ±{s:.2f}" if np.isfinite(m) else "" for m, s in zip(PROFILE["mean"], PROFILE["sd"])])
print("mean ± SD of the keyed ratings (1-5), per run, prompt and factor:\n")
show(wide.pivot(index=["run", "prompt"], columns="factor", values="cell").reindex(columns=FACTORS)
     .reindex([(n, c) for n in RUN_NAMES for c in CONDITIONS]), index=True)

theta = np.linspace(0, 2 * np.pi, len(FACTORS), endpoint=False)
loop = lambda v: np.concatenate([v, v[:1]])
DAN_LABEL = ["DAN0\nright away", "DAN1\nanswer", "DAN2\ntake your time"]
INFO_LABEL = ["INFO0: HEXACO test", "INFO1: profiling question", "INFO2: question"]
for name in RUN_NAMES:
    fig, axes = plt.subplots(3, 3, figsize=(10, 10.6), subplot_kw={"projection": "polar"})
    for (d, i), ci in COND_AT.items():
        ax = axes[d, i]
        mean = MEAN[name][ci]
        spread = SD[name][ci] if SPREAD == "sd" else SD[name][ci] ** 2
        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)
        ax.set_rorigin(0)
        ax.set_ylim(1, 5)
        ax.set_yticks([1, 2, 3, 4, 5], ["1", "", "3", "", "5"], fontsize=7, color=MUTED)
        ax.set_rlabel_position(30)
        ax.set_xticks(theta, FACTORS, fontsize=10, color=INK)
        ax.spines["polar"].set_color(BASELINE)
        ax.plot(np.linspace(0, 2 * np.pi, 100), np.full(100, 3), color=BASELINE, linewidth=1.2, zorder=1)
        ok = np.isfinite(mean)
        if ok.all():
            ax.fill_between(loop(theta), loop(np.clip(mean - spread, 1, 5)), loop(np.clip(mean + spread, 1, 5)),
                            color=BLUE, alpha=0.18, linewidth=0, zorder=2)
            ax.plot(loop(theta), loop(mean), color=BLUE, linewidth=2, marker="o", markersize=4.5, zorder=4)
        else:   # a factor with no rated item: dots only, no closed shape
            ax.errorbar(theta[ok], mean[ok], yerr=spread[ok], color=BLUE, fmt="o", markersize=4.5, zorder=4)
        if ci != BASE and np.isfinite(MEAN[name][BASE]).all():
            ax.plot(loop(theta), loop(MEAN[name][BASE]), color=MUTED, linewidth=1.2, linestyle=(0, (3, 2)), zorder=3)
        rated = int(RATED[name][ci].sum())
        ax.set_title(f"{CONDITIONS[ci]}{' (baseline)' if ci == BASE else ''}   {rated}/{len(ITEMS)} rated",
                     fontsize=9, loc="center", pad=14, fontweight="bold" if ci == BASE else "normal")
    for i in range(3):
        fig.text(0.215 + 0.3 * i, 0.925, INFO_LABEL[i], ha="center", fontsize=10.5, color=INK2, fontweight="bold")
    for d in range(3):
        fig.text(0.012, 0.745 - 0.28 * d, DAN_LABEL[d], ha="left", va="center", fontsize=10.5, color=INK2, fontweight="bold")
    fig.legend(handles=[Line2D([], [], color=BLUE, marker="o", markersize=4.5, label="mean keyed rating"),
                        Patch(color=BLUE, alpha=0.18, label="± 1 SD over the factor's items" if SPREAD == "sd"
                              else "± the variance over the factor's items"),
                        Line2D([], [], color=MUTED, linewidth=1.2, linestyle=(0, (3, 2)), label="baseline D1I2"),
                        Line2D([], [], color=BASELINE, linewidth=1.2, label="neutral (3)")],
               loc="lower center", ncol=4, fontsize=9)
    fig.suptitle(f"{name}: personality profile under the nine prompts", x=0.012, y=0.985, ha="left", fontsize=12, fontweight="bold")
    fig.subplots_adjust(left=0.1, right=0.97, top=0.85, bottom=0.07, hspace=0.45, wspace=0.3)
    save(fig, f"1_profile_{name}")
    plt.show()

## 2. Answers, and what the capture cost

Per run: capture time and speed (`run_info.json`), reply lengths (`responses.json`), and how the ratings were
read: directly (`parsed`), by the judge, or not at all. A reply that reaches `max_new_tokens` was cut off; for a
thinking run that usually means it never closed `</think>` and has no answer. The judge check is the judge's
agreement with directly-read ratings it was also shown.

The heatmaps are per run and prompt: the share of items rated (a prompt with few rated items has weak scores
everywhere below), and the mean reply length (DAN2, "take your time", is expected to be the long one).

In [ ]:
cost, per_prompt = [], []
for name in RUN_NAMES:
    info, conds = RUN_INFO[name], RUN_INFO[name].get("conditions", {})
    lengths = [r["tokens"] for c in CONDITIONS for r in RESPONSES[name][c]]
    flat = [r for c in CONDITIONS for r in RATINGS[name][c]]
    by = Counter(r["by"] for r in flat)
    check = RATINGS[name].get("judge check", [])
    cap = info.get("max_new_tokens")
    cost.append({
        "run": name, "dtype": info.get("dtype"), "max new tokens": cap,
        "minutes": sum(c["seconds"] for c in conds.values() if "seconds" in c) / 60
                   if any("seconds" in c for c in conds.values()) else np.nan,
        "gen tok/s": np.mean([c["gen tok/s"] for c in conds.values() if "gen tok/s" in c])
                     if any("gen tok/s" in c for c in conds.values()) else np.nan,
        "mean reply tokens": np.mean(lengths), "replies at the cap": np.mean([n >= cap for n in lengths]) if cap else np.nan,
        "never finished thinking": sum(c.get("replies that never finished thinking", 0) for c in conds.values())
                                   if any("replies that never finished thinking" in c for c in conds.values()) else None,
        "non-finite captures": sum(c.get("non-finite captures", 0) for c in conds.values()),
        "rated": np.isfinite(RAW[name]).mean(), "parsed": by.get("parsed", 0) / len(flat),
        "by judge": sum(v for k, v in by.items() if k not in ("parsed", "empty")) / len(flat),
        "judge errors": sum(str(r["rating"]).startswith("ERROR") for r in flat),
        "judge check": f"{sum(c['parsed'] == c['judge'] for c in check)}/{len(check)}"})
    for ci, c in enumerate(CONDITIONS):
        per_prompt.append({"run": name, "prompt": c, "rated": np.isfinite(RAW[name][ci]).mean(),
                           "mean reply tokens": np.mean([r["tokens"] for r in RESPONSES[name][c]]),
                           "seconds": conds.get(c, {}).get("seconds"),
                           "at the cap": conds.get(c, {}).get("replies at the token limit")})
COST = pd.DataFrame(cost).astype({"never finished thinking": "Int64"})
show(COST, {"minutes": "{:.0f}".format, "gen tok/s": "{:.1f}".format, "mean reply tokens": "{:.0f}".format,
            "replies at the cap": pct, "rated": pct, "parsed": pct, "by judge": pct})
print("\nThe minutes cover the conditions captured in the last session (an earlier session's keep their own seconds).")
COST.to_csv(RESULTS / "cost.csv", index=False)
ANSWERS = pd.DataFrame(per_prompt)
ANSWERS.to_csv(RESULTS / "answers.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 0.42 * len(RUN_NAMES) + 1.6), gridspec_kw={"width_ratios": [1, 1.25, 1.25]})
ax = axes[0]
y = np.arange(len(COST))[::-1]
ax.barh(y, COST["minutes"].fillna(0), color=BLUE, height=0.6)
for yi, m in zip(y, COST["minutes"]):
    ax.text(0 if pd.isna(m) else m, yi, "  no timing" if pd.isna(m) else f"  {m:.0f} min", va="center", color=INK2, fontsize=8)
ax.set_yticks(y, COST["run"])
ax.set_xlabel("capture minutes, nine prompts (generation + hidden-state read)")
ax.set_title("Capture time per run")
ax.grid(axis="y", visible=False)
ax.set_xlim(0, COST["minutes"].max() * 1.4 if COST["minutes"].max() > 0 else 1)
rated = ANSWERS.pivot(index="run", columns="prompt", values="rated").reindex(index=RUN_NAMES, columns=CONDITIONS)
heat(axes[1], rated, SEQ, 0, 1, lambda v: f"{v:.0%}")
axes[1].set_title("Items rated")
tokens = ANSWERS.pivot(index="run", columns="prompt", values="mean reply tokens").reindex(index=RUN_NAMES, columns=CONDITIONS)
heat(axes[2], tokens, SEQ, 0, max(float(np.nanmax(tokens.to_numpy())), 1), lambda v: f"{v:.0f}")
axes[2].set_title("Mean reply tokens")
axes[2].set_yticks([])
fig.tight_layout()
save(fig, "2_cost_and_answers")
plt.show()

## 3. Stable and unstable traits

A trait is **stable** in a model if its score holds still while the prompt changes, and **unstable** if the
differential prompts move it. Per model and factor, over the nine prompts:

- **range** and **SD across prompts** of the factor's score: how far the prompts move the whole factor.
- **item SD across prompts**: for each item rated under all nine prompts, the SD of its own nine ratings,
  averaged over the factor's items. It catches a factor whose items move in opposite ways and cancel in the score.
- **items unchanged**: the share of those items with the same rating under every prompt.
- **prompts that move it**: of the eight non-baseline prompts, how many shift the factor by more than item
  noise: the 95% bootstrap interval (over items) of the paired shift, prompt minus baseline, excludes zero.
  This is the count to read stability from; it needs no cut-off on the size of a shift. With eight intervals at
  95%, one false alarm per three stable factors is expected, so 0-1 reads as stable and several as unstable.
- **between / within**: the variance of the factor's score across prompts, over the mean item variance within a
  prompt (section 1's band, squared). Small means the prompt matters little next to how unevenly the model
  rates the factor's items anyway.

First chart: the range per model and factor, with the count of prompts that move it. Second: each factor's
score under every prompt (ticks), the span they cover, and the baseline (dot).

In [ ]:
def boot_ci(values, seed=BOOT_SEED):
    # 95% bootstrap interval of the mean, resampling the values (one per item); (nan, nan) under 3 values.
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 3:
        return np.nan, np.nan
    draws = np.random.default_rng(seed).integers(0, len(values), (BOOT, len(values)))
    return tuple(np.percentile(values[draws].mean(1), [2.5, 97.5]))


def paired(name, a, b, items):
    # Per item rated under both conditions: keyed[a] - keyed[b].
    d = KEYED[name][a][items] - KEYED[name][b][items]
    return d[np.isfinite(d)]


SHIFT_ROWS = []   # every non-baseline prompt against the baseline, per run and factor
for name in RUN_NAMES:
    for c in OTHERS:
        for f in FACTORS:
            d = paired(name, CONDITIONS.index(c), BASE, ITEMS_OF[f])
            lo, hi = boot_ci(d)
            SHIFT_ROWS.append({"run": name, "prompt": c, "factor": f, "items": len(d),
                               "shift": d.mean() if len(d) else np.nan, "low": lo, "high": hi,
                               "moved": bool(lo > 0 or hi < 0)})
SHIFTS = pd.DataFrame(SHIFT_ROWS)
SHIFTS.to_csv(RESULTS / "shifts.csv", index=False)

rows = []
for name in RUN_NAMES:
    for fi, f in enumerate(FACTORS):
        scores = MEAN[name][:, fi]
        k = KEYED[name][:, ITEMS_OF[f]]
        full_items = k[:, np.isfinite(k).all(0)]
        mine = SHIFTS[(SHIFTS["run"] == name) & (SHIFTS["factor"] == f)]
        within = np.nanmean(SD[name][:, fi] ** 2)
        rows.append({"run": name, "factor": f, "baseline": scores[BASE], "min": np.nanmin(scores), "max": np.nanmax(scores),
                     "range": np.nanmax(scores) - np.nanmin(scores), "SD across prompts": np.nanstd(scores),
                     "item SD across prompts": full_items.std(0).mean() if full_items.shape[1] else np.nan,
                     "items unchanged": np.mean(np.ptp(full_items, axis=0) == 0) if full_items.shape[1] else np.nan,
                     "items in all prompts": full_items.shape[1],
                     "prompts that move it": int(mine["moved"].sum()), "of": len(mine),
                     "largest shift": mine.loc[mine["shift"].abs().idxmax(), "shift"] if mine["shift"].notna().any() else np.nan,
                     "at": mine.loc[mine["shift"].abs().idxmax(), "prompt"] if mine["shift"].notna().any() else "",
                     "between / within": np.nanvar(scores) / within if within > 0 else np.nan})
STABILITY = pd.DataFrame(rows)
show(STABILITY, {c: two for c in ("baseline", "min", "max", "range", "SD across prompts", "item SD across prompts",
                                  "between / within")} | {"items unchanged": pct, "largest shift": num})
STABILITY.to_csv(RESULTS / "stability.csv", index=False)

print("\nmost stable and least stable factor per run (by prompts that move it, then range):")
for name in RUN_NAMES:
    s = STABILITY[STABILITY["run"] == name].sort_values(["prompts that move it", "range"])
    a, b = s.iloc[0], s.iloc[-1]
    print(f"  {name}: most stable {a['factor']} ({a['prompts that move it']}/{a['of']} prompts, range {a['range']:.2f}); "
          f"least stable {b['factor']} ({b['prompts that move it']}/{b['of']} prompts, range {b['range']:.2f}, "
          f"largest {b['largest shift']:+.2f} at {b['at']})")
print("\nper factor over all runs (mean):")
show(STABILITY.groupby("factor", sort=False)[["range", "SD across prompts", "item SD across prompts", "items unchanged",
                                              "prompts that move it"]].mean().reset_index(),
     {"range": two, "SD across prompts": two, "item SD across prompts": two, "items unchanged": pct,
      "prompts that move it": "{:.1f}".format})

fig, ax = plt.subplots(figsize=(1.0 * len(FACTORS) + 2.4, 0.42 * len(RUN_NAMES) + 1.3))
table = STABILITY.pivot(index="run", columns="factor", values="range").reindex(index=RUN_NAMES, columns=FACTORS)
moved = STABILITY.pivot(index="run", columns="factor", values="prompts that move it").reindex(index=RUN_NAMES, columns=FACTORS)
vmax = max(float(np.nanmax(table.to_numpy())), 0.5)
ax.imshow(table.to_numpy(dtype=float), cmap=SEQ, vmin=0, vmax=vmax, aspect="auto")
for i in range(table.shape[0]):
    for j in range(table.shape[1]):
        v = table.iloc[i, j]
        if np.isfinite(v):
            ax.text(j, i, f"{v:.2f}\n{int(moved.iloc[i, j])}/{len(OTHERS)}", ha="center", va="center", fontsize=8,
                    color="white" if v > 0.6 * vmax else INK)
ax.set_xticks(range(len(FACTORS)), FACTORS)
ax.set_yticks(range(len(RUN_NAMES)), RUN_NAMES)
ax.grid(False)
ax.tick_params(length=0)
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("Range of each factor's score over the nine prompts\n(below: prompts that move it beyond item noise)")
fig.tight_layout()
save(fig, "3_stability_range")
plt.show()

fig, axes = grid_of(len(RUN_NAMES), per_row=4, width=3.3, height=2.9, sharey=True)
x = np.arange(len(FACTORS))
for ax, name in zip(axes, RUN_NAMES):
    lo, hi = np.nanmin(MEAN[name], 0), np.nanmax(MEAN[name], 0)
    ax.axhline(3, color=BASELINE, linewidth=1)
    ax.vlines(x, lo, hi, color=BLUE, linewidth=5, alpha=0.3)
    for ci in range(len(CONDITIONS)):
        ax.scatter(x, MEAN[name][ci], marker="_", s=110, color=BLUE, linewidth=1.2, zorder=3)
    ax.scatter(x, MEAN[name][BASE], color=INK, s=22, zorder=4)
    ax.set_xticks(x, FACTORS)
    ax.set_ylim(1, 5)
    ax.set_xlim(-0.6, len(FACTORS) - 0.4)
    ax.grid(axis="x", visible=False)
    ax.set_title(name, fontsize=9)
axes[0].set_ylabel("factor score (1-5, keyed)")
fig.legend(handles=[Line2D([], [], color=BLUE, marker="_", linestyle="", markersize=10, label="one prompt"),
                    Patch(color=BLUE, alpha=0.3, label="span over the nine prompts"),
                    Line2D([], [], color=INK, marker="o", linestyle="", markersize=5, label="baseline D1I2")],
           loc="lower center", ncol=3, fontsize=8)
fig.suptitle("Where each factor's score sits under the nine prompts", x=0.01, ha="left", fontsize=11, fontweight="bold")
fig.tight_layout(rect=(0, 0.06, 1, 0.95))
save(fig, "3_stability_spans")
plt.show()

## 4. Shifts from the prompts

Every shift is **paired by item**: over the items rated under both prompts, the mean of (keyed rating under one
minus under the other), in points on the 1-5 scale.

- **Per prompt (heatmaps):** each non-baseline prompt against the baseline D1I2, per model. A `*` marks a shift
  whose 95% bootstrap interval excludes zero.
- **Main effects (dot chart):** one axis at a time, at the same level of the other axis: DAN0 and DAN2 against
  DAN1, INFO0 and INFO1 against INFO2. Pooled over models and over the other axis's three levels, with a 95%
  bootstrap interval over the item-level differences (the models are the ones run, not a sample of models).
- **Overall:** how much each prompt moves a profile, as the mean absolute shift over the six factors, and
  whether DAN and INFO add up: the shift of a corner prompt (say D0I0) against the sum of its two single-axis
  shifts (D0I2 + D1I0).

The earlier black-box study's findings to check: full task knowledge (INFO0) raised N and often C; DAN lowered
C and moved A; small models are bland (scores near 3, little movement).

In [ ]:
vmax = max(float(SHIFTS["shift"].abs().max()), 0.5)
fig, axes = grid_of(len(RUN_NAMES), per_row=4, width=3.4, height=3.0)
for ax, name in zip(axes, RUN_NAMES):
    mine = SHIFTS[SHIFTS["run"] == name]
    table = mine.pivot(index="prompt", columns="factor", values="shift").reindex(index=OTHERS, columns=FACTORS)
    marks = mine.pivot(index="prompt", columns="factor", values="moved").reindex(index=OTHERS, columns=FACTORS)
    heat(ax, table, DIV, -vmax, vmax, lambda v: f"{v:+.1f}", marks)
    ax.set_title(name, fontsize=9)
fig.suptitle(f"Shift against the baseline {DESIGN['baseline']}, in points (blue = lower, red = higher; * = interval excludes 0)",
             x=0.01, ha="left", fontsize=11, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.94))
save(fig, "4_shifts_per_prompt")
plt.show()

EFFECT_DEFS = [("DAN", 0, 1), ("DAN", 2, 1), ("INFO", 0, 2), ("INFO", 1, 2)]
EFFECT_TEXT = {("DAN", 0): "right away, without thinking", ("DAN", 2): "take your time and think",
               ("INFO", 0): "told it is a HEXACO test", ("INFO", 1): "told it is a profiling question"}


def effect_diffs(name, axis, level, ref, f):
    # Item-level paired differences for one main effect, over the other axis's levels.
    out = []
    for o in range(3):
        a, b = ((level, o), (ref, o)) if axis == "DAN" else ((o, level), (o, ref))
        if a in COND_AT and b in COND_AT:
            out.append(paired(name, COND_AT[a], COND_AT[b], ITEMS_OF[f]))
    return np.concatenate(out) if out else np.zeros(0)


rows = []
for axis, level, ref in EFFECT_DEFS:
    for f in FACTORS:
        per_run = {name: effect_diffs(name, axis, level, ref, f) for name in RUN_NAMES}
        for name, d in list(per_run.items()) + [("all runs", np.concatenate(list(per_run.values())))]:
            lo, hi = boot_ci(d)
            rows.append({"effect": f"{axis}{level} vs {axis}{ref}", "run": name, "factor": f, "pairs": len(d),
                         "shift": d.mean() if len(d) else np.nan, "low": lo, "high": hi})
EFFECTS = pd.DataFrame(rows)
EFFECTS.to_csv(RESULTS / "effects.csv", index=False)
print("main effects per run (points, paired by item):\n")
show(EFFECTS[EFFECTS["run"] != "all runs"].pivot(index=["effect", "run"], columns="factor", values="shift")
     .reindex(columns=FACTORS).reindex([(f"{a}{l} vs {a}{r}", n) for a, l, r in EFFECT_DEFS for n in RUN_NAMES]),
     {f: num for f in FACTORS}, index=True)
pooled = EFFECTS[EFFECTS["run"] == "all runs"]
print("\npooled over runs, with 95% intervals:\n")
show(pooled.assign(interval=[f"[{a:+.2f}, {b:+.2f}]" for a, b in zip(pooled["low"], pooled["high"])])
     [["effect", "factor", "pairs", "shift", "interval"]], {"shift": num})

fig, axes = plt.subplots(1, 4, figsize=(13, 3.1), sharey=True, sharex=True)
y = np.arange(len(FACTORS))[::-1]
for ax, (axis, level, ref) in zip(axes, EFFECT_DEFS):
    mine = pooled[pooled["effect"] == f"{axis}{level} vs {axis}{ref}"].set_index("factor").reindex(FACTORS)
    ax.axvline(0, color=BASELINE, linewidth=1)
    for name in RUN_NAMES:   # each run, faint, behind the pooled estimate
        per = EFFECTS[(EFFECTS["effect"] == f"{axis}{level} vs {axis}{ref}") & (EFFECTS["run"] == name)].set_index("factor").reindex(FACTORS)
        ax.scatter(per["shift"], y, color=MUTED, s=10, alpha=0.6, zorder=2)
    ax.hlines(y, mine["low"], mine["high"], color=BLUE, linewidth=2, zorder=3)
    ax.scatter(mine["shift"], y, color=BLUE, s=36, zorder=4, edgecolor=SURFACE, linewidth=1)
    ax.grid(axis="y", visible=False)
    ax.set_title(f"{axis}{level} vs {axis}{ref}\n{EFFECT_TEXT[axis, level]}", fontsize=9)
    ax.set_xlabel("shift in points (1-5 scale)")
axes[0].set_yticks(y, [f"{hx.FACTORS[f]}  {f}" for f in FACTORS])
fig.legend(handles=[Line2D([], [], color=BLUE, marker="o", markersize=6, label="all runs pooled, 95% interval"),
                    Line2D([], [], color=MUTED, marker="o", linestyle="", markersize=3.5, label="one run")],
           loc="lower center", ncol=2, fontsize=8)
fig.tight_layout(rect=(0, 0.07, 1, 1))
save(fig, "4_main_effects")
plt.show()

overall = SHIFTS.assign(size=SHIFTS["shift"].abs()).pivot_table(index="run", columns="prompt", values="size", aggfunc="mean") \
    .reindex(index=RUN_NAMES, columns=OTHERS)
overall.loc["all runs"] = overall.mean()
overall["mean"] = overall.mean(axis=1)
print("\nhow much each prompt moves a profile: mean |shift| over the six factors, in points:\n")
show(overall, {c: two for c in overall}, index=True)
overall.to_csv(RESULTS / "shift_sizes.csv")

rows = []
for name in RUN_NAMES:
    get = lambda c: SHIFTS[(SHIFTS["run"] == name) & (SHIFTS["prompt"] == c)].set_index("factor")["shift"].reindex(FACTORS)
    bd, bi = LEVELS[DESIGN["baseline"]]
    for c in OTHERS:
        d, i = LEVELS[c]
        if d != bd and i != bi and f"D{d}I{bi}" in OTHERS and f"D{bd}I{i}" in OTHERS:
            both, added = get(c), get(f"D{d}I{bi}") + get(f"D{bd}I{i}")
            rows.append({"run": name, "prompt": c, "rho, joint vs sum": hx.spearman(both, added),
                         "mean |joint - sum|": float((both - added).abs().mean())})
ADDITIVE = pd.DataFrame(rows)
if len(ADDITIVE):
    print("\ndo DAN and INFO add up? A corner prompt's shift against the sum of its two single-axis shifts, over the factors:\n")
    show(ADDITIVE, {"rho, joint vs sum": num, "mean |joint - sum|": two})
    ADDITIVE.to_csv(RESULTS / "additivity.csv", index=False)

## Hidden states

From here on the analysis is white-box. `slices(run, kind)` gives `[conditions, items, hidden]` at one layer:
`LAYER`, or 20/28 of the model's depth, persona-lsh's default. `classify` is persona-lsh's nearest-mean
classifier: cosine to each class mean, and Hamming distance between RP-LSH digests at each size in `RP_BITS`.

In [ ]:
N_LAYERS, _SLICES, _HASHERS = {}, {}, {}


def default_layer(n_layers):
    return LAYER if LAYER is not None else max(1, round(n_layers * 20 / 28))


def full(name, kind):
    # [conditions, items, layers+1, hidden] as stored (fp16 or fp32), or None if the run has no such kind.
    per = []
    for c in CONDITIONS:
        saved = torch.load(OUT_DIR / name / f"states_{c}.pt", map_location="cpu")
        if kind not in saved:
            return None
        per.append(saved[kind].numpy())
    N_LAYERS[name] = per[0].shape[1] - 1
    return np.stack(per)


def slices(name, kind):
    if (name, kind) not in _SLICES:
        per = []
        for c in CONDITIONS:
            saved = torch.load(OUT_DIR / name / f"states_{c}.pt", map_location="cpu")
            if kind not in saved:
                per = None
                break
            N_LAYERS[name] = saved[kind].shape[1] - 1
            per.append(saved[kind][:, default_layer(N_LAYERS[name])].float().numpy())
        _SLICES[name, kind] = None if per is None else np.stack(per)
    return _SLICES[name, kind]


def finite(a):
    return a[np.isfinite(a).all(-1)]


def hasher(dim, bits):
    if (dim, bits) not in _HASHERS:
        _HASHERS[dim, bits] = RandomProjectionBackend(dim=dim, n_bits=bits, seed=RP_SEED)
    return _HASHERS[dim, bits]


METHODS = ["cosine"] + [f"rp {b}" for b in RP_BITS]


def classify(train, test, rp=True):
    # One row per test vector: true label and the label each method picks.
    train = {k: finite(v) for k, v in train.items()}
    test = {k: finite(v) for k, v in test.items()}
    if any(len(v) == 0 for v in train.values()):
        return []
    dim = next(iter(train.values())).shape[1]
    rows = [{"true": r["true"], "cosine": r["cosine_pred"]} for r in hx.nearest_mean(train, test)]
    for bits in (RP_BITS if rp else []):
        for row, got in zip(rows, hx.nearest_mean(train, test, hasher(dim, bits))):
            row[f"rp {bits}"] = got["rp_pred"]
    return rows


def cross_validate(vectors, rp=True):
    # CV_FOLDS folds within each class; the rows of every fold's held-out part.
    vectors = {k: finite(v) for k, v in vectors.items()}
    parts = {k: hx.folds(len(v), CV_FOLDS, f"{CV_SEED}/{k}") for k, v in vectors.items()}
    rows = []
    for f in range(CV_FOLDS):
        rows += classify({k: v[parts[k][f][0]] for k, v in vectors.items()},
                         {k: v[parts[k][f][1]] for k, v in vectors.items()}, rp)
    return rows


def acc(rows, prefix=""):
    return {f"{prefix}n": len(rows)} | {f"{prefix}{m}": (sum(r[m] == r["true"] for r in rows) / len(rows) if rows else np.nan)
                                        for m in METHODS if not rows or m in rows[0]}


for name in RUN_NAMES:
    got = {kind: slices(name, kind) for kind in KINDS}
    print(f"{name}: {N_LAYERS[name]} layers, layer {default_layer(N_LAYERS[name])}, "
          + ", ".join(f"{k} {'missing' if v is None else v.shape}" for k, v in got.items()))

## 5. Factor identification: which factor is the item about?

The persona-lsh classification, with the six factors as the classes. Under the baseline prompt, each item's
vector is centered on the train items' mean, the six factor means come from the train half, and items are
classified by the nearest mean: cross-validated within the train half (`cv`), and on the eval half (`held-out`).
Chance is 1/6.

Then the same baseline means classify the eval items **under the other eight prompts**: `other prompts` keeps
the baseline's centre (a library built once, at baseline), `own centre` re-centers on each prompt's own train
mean. That is the robustness check of PLAN section 4: do digests survive a differential prompt?

This test is about the item's **content**: a model that answered every item "3" could still pass it. It shows
that the states and digests separate the six factors' item sets and it bounds what RP-LSH loses. Whether the
states carry the model's own standing on a factor is section 6.

In [ ]:
def factor_sets(X, cond=None, centre="baseline"):
    # (train, test) dicts by factor at the baseline. With cond, test = the eval items under that condition.
    mu = X[BASE][ALL_TRAIN]
    mu = finite(mu).mean(0)
    train = {f: X[BASE][TRAIN[f]] - mu for f in FACTORS}
    if cond is None:
        return train, {f: X[BASE][EVAL[f]] - mu for f in FACTORS}
    if centre == "own":
        mu = finite(X[cond][ALL_TRAIN]).mean(0)
    return train, {f: X[cond][EVAL[f]] - mu for f in FACTORS}


rows = []
for name in RUN_NAMES:
    for kind in KINDS:
        X = slices(name, kind)
        if X is None:
            continue
        train, test = factor_sets(X)
        row = {"run": name, "kind": kind, "chance": 1 / len(FACTORS)}
        row |= acc(cross_validate(train), "cv ") | acc(classify(train, test), "held-out ")
        for centre, label in (("baseline", "other prompts "), ("own", "own centre ")):
            per = [acc(classify(*factor_sets(X, CONDITIONS.index(c), centre)))["cosine"] for c in OTHERS]
            row[label + "cosine"] = np.mean(per) if per else np.nan
            row[label + "min"] = np.min(per) if per else np.nan
        rows.append(row)
FACTOR_ID = pd.DataFrame(rows)
show(FACTOR_ID, {c: pct for c in FACTOR_ID if c not in ("run", "kind") and not c.endswith(" n")})
FACTOR_ID.to_csv(RESULTS / "factor_identification.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(1.6 * len(RUN_NAMES) + 2.5, 3.2))
kind = "prompt" if "prompt" in KINDS else KINDS[0]
mine = FACTOR_ID[FACTOR_ID["kind"] == kind].set_index("run").reindex(RUN_NAMES)
series = [("held-out cosine", "cosine"), *[(f"held-out rp {b}", f"RP-LSH {b} bits") for b in sorted(RP_BITS, reverse=True)]]
width = 0.8 / len(series)
for k, (column, label) in enumerate(series):
    x = np.arange(len(mine)) + (k - (len(series) - 1) / 2) * width
    ax.bar(x, mine[column], width=width * 0.92, color=SLOTS[k], label=label, edgecolor=SURFACE, linewidth=1)
    for xi, v in zip(x, mine[column]):
        if np.isfinite(v):
            ax.text(xi, v + 0.01, f"{v:.0%}", ha="center", va="bottom", fontsize=7, color=INK2)
ax.axhline(1 / len(FACTORS), color=MUTED, linewidth=1, linestyle=(0, (3, 2)))
ax.text(len(mine) - 0.45, 1 / len(FACTORS) + 0.01, "chance", fontsize=8, color=MUTED, ha="right")
ax.set_xticks(range(len(mine)), mine.index)
ax.set_ylim(0, 1.1)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.grid(axis="x", visible=False)
ax.legend(loc="upper left", ncol=len(series), fontsize=8, bbox_to_anchor=(0, 1.02))
ax.set_title(f"Factor identification on the held-out items, baseline prompt ({kind} vectors)", pad=22)
fig.tight_layout()
save(fig, "5_factor_identification")
plt.show()

## 6. Personality direction: does a factor's direction predict the held-out answers?

For each factor, the direction along which the model's own answers vary. Take every rated answer to a **train**
item of the factor, under all nine prompts. Center each vector on its prompt's mean over those train items (so
neither the prompt's wording nor the factor's topic is in it), and weight it by how far its keyed rating is
from the average: `direction = sum of (keyed rating - mean) x (vector - centre)`. With answers of only "agree"
and "disagree" this is the difference of the two groups' means, persona-lsh's with minus without, read off the
model's own answers instead of an instruction.

Tested on the **eval** half, which had no part in the direction:
- **rho, factor direction**: Spearman correlation between an answer's projection on the direction and its
  keyed rating, over all eval answers of the factor.
- **rho, agreement direction**: the control. One direction for raw agreement (rating 1-5 as given), built from
  every factor's train items, with its projection flipped for reverse-keyed items. If this does as well, the
  factor direction is just "the model is about to agree", not the factor.
- **rho, same item across prompts**: both projection and rating centered per item, over eval items rated under
  two or more prompts. Item content cancels, so this asks whether the state moves with the answer when only
  the prompt changes. It is empty when a model gives each item the same rating under every prompt.

`score SD` is the spread of the train answers: a model that answers a factor nearly uniformly has no direction
to find (blank row). `prompt` is the fair kind here: `response` has seen the digit it wrote.

In [ ]:
SIGN = np.where(REVERSE, -1.0, 1.0)
DIRECTIONS = {}   # (run, kind) -> {factor: unit direction}


def centres(X, ok, items):
    # Per condition, the mean vector over the usable answers among `items` (None if there are none).
    # `ok` [conditions, items] says which answers count: pass a mask limited to some conditions to use only those.
    return [X[c][items[ok[c][items]]].mean(0) if ok[c][items].any() else None for c in range(len(CONDITIONS))]


def points(X, ok, values, items, centre):
    # Every usable answer among `items`: (vector - its condition's centre, value, item index).
    xs, ys, ids = [], [], []
    for c in range(len(CONDITIONS)):
        use = items[ok[c][items]]
        if centre[c] is not None and len(use):
            xs.append(X[c][use] - centre[c])
            ys.append(values[c][use])
            ids.append(use)
    if not xs:
        return np.zeros((0, X.shape[-1])), np.zeros(0), np.zeros(0, dtype=int)
    return np.concatenate(xs), np.concatenate(ys), np.concatenate(ids)


def within(values, ids):
    # values minus their item's mean, for items with two or more values; (centered values, mask).
    out, keep = np.zeros(len(values)), np.zeros(len(values), dtype=bool)
    for i in np.unique(ids):
        same = ids == i
        if same.sum() >= 2:
            out[same], keep[same] = values[same] - values[same].mean(), True
    return out[keep], keep


def personality_directions(X, raw, keyed):
    # ({factor: unit direction}, rows of the test on the eval half)
    ok = np.isfinite(X).all(-1) & np.isfinite(raw)
    all_centre = centres(X, ok, ALL_TRAIN)
    agree = hx.covariance_direction(*points(X, ok, raw, ALL_TRAIN, all_centre)[:2])
    directions, rows = {}, []
    for f in FACTORS:
        centre = centres(X, ok, TRAIN[f])
        tx, ty, _ = points(X, ok, keyed, TRAIN[f], centre)
        ex, ey, ei = points(X, ok, keyed, EVAL[f], centre)
        row = {"factor": f, "train answers": len(ty), "eval answers": len(ey),
               "score SD": ty.std() if len(ty) else np.nan}
        d = hx.covariance_direction(tx, ty)
        if d is not None and np.linalg.norm(d) > 0:
            d = directions[f] = d / np.linalg.norm(d)
            proj = ex @ d
            row["rho, factor direction"] = hx.spearman(proj, ey)
            (p, keep), (y, _) = within(proj, ei), within(ey, ei)
            row["rho, same item across prompts"] = hx.spearman(p, y)
            row["items in it"] = len(np.unique(ei[keep]))
        if agree is not None:
            gx, gy, gi = points(X, ok, keyed, EVAL[f], all_centre)
            row["rho, agreement direction"] = hx.spearman((gx @ agree) * SIGN[gi], gy)
        rows.append(row)
    return directions, rows


rows = []
for name in RUN_NAMES:
    for kind in KINDS:
        X = slices(name, kind)
        if X is None:
            continue
        DIRECTIONS[name, kind], got = personality_directions(X, RAW[name], KEYED[name])
        rows += [{"run": name, "kind": kind} | r for r in got]
PERSONALITY = pd.DataFrame(rows)
rho_cols = [c for c in PERSONALITY if c.startswith("rho")]
show(PERSONALITY, {c: num for c in rho_cols} | {"score SD": lambda v: "" if pd.isna(v) else f"{v:.2f}"})
PERSONALITY.to_csv(RESULTS / "personality_direction.csv", index=False)

print("\nmedian over factors, per run and kind:\n")
show(PERSONALITY.groupby(["run", "kind"], sort=False)[rho_cols].median().reset_index(), {c: num for c in rho_cols})

In [ ]:
GEO_KIND = "prompt" if "prompt" in KINDS else KINDS[0]
mine = PERSONALITY[PERSONALITY["kind"] == GEO_KIND]
fig, axes = plt.subplots(1, 2, figsize=(11, 0.42 * len(RUN_NAMES) + 1.5), gridspec_kw={"width_ratios": [1, 1]})
for ax, column, title in ((axes[0], "rho, factor direction", "Factor direction"),
                          (axes[1], "rho, agreement direction", "Control: agreement direction")):
    if column in mine:
        table = mine.pivot(index="run", columns="factor", values=column).reindex(index=RUN_NAMES, columns=FACTORS)
        heat(ax, table.astype(float), DIV, -1, 1, lambda v: f"{v:+.2f}")
    ax.set_title(f"{title}: rho with the held-out ratings ({GEO_KIND})", fontsize=9)
axes[1].set_yticks([])
fig.tight_layout()
save(fig, "6_personality_direction")
plt.show()

## 7. Direction and magnitude: separable or confused?

A persona vector has a **direction** (which trait) and a **magnitude** (how much of it). Two questions, both on
the eval half with directions from the train half, both per model:

**7a. Are the six directions separable from each other?** Project the eval answers of factor *g* on the
direction of factor *f*, and correlate with *g*'s keyed ratings, for every pair. The diagonal (*f* = *g*) is
section 6's rho. Off the diagonal, a direction that also predicts another factor's answers is **confused** with
it. Three things can produce that, and the tables tell them apart: the two factors really are related in this
model (H with A, N against E and C: the cosine table shows related directions), both directions are mostly the
agreement direction (section 6's control is then as good as the factor direction), or the direction is noise
(the diagonal is near zero too).
- `own factor`: the median diagonal rho. `other factors`: the median absolute off-diagonal rho. A rank
  correlation says whether *f*'s direction responds to *g* at all, not how strongly: with little noise even a
  small overlap gives a large rho.
- `leakage` says how strongly: the covariance of *g*'s ratings with the projection on *f*'s direction, over
  their covariance with the projection on *g*'s own direction. 0: none of *g*'s signal lands on *f*'s
  direction. 1: as much as on its own, so the two can't be told apart. The summary is the median absolute
  leakage over the pairs, with a 95% interval from resampling the eval items.
- **Verdict:** *no direction* if the interval of `own factor` includes zero. Otherwise *separable* if the
  whole interval of the leakage is under one half, *confused* if all of it is over one half, and *partly
  confused* if it straddles. One half is a wide line on purpose: read the number, and the heatmap for which
  pairs carry it.

**7b. Within a factor, is magnitude separable from direction?** For an eval answer, with its vector centered
as in section 6: the **projection** on the factor's direction uses both angle and length; the **cosine** to the
direction is angle only; the vector's **length** is magnitude with no direction at all.
- `pole accuracy`: the sign of the projection against whether the rating is above the factor's average. One
  bit, as an RP-LSH digest keeps: which side of the trait the answer is on.
- `rho within a pole`: among answers on the same side, does a larger projection still mean a more extreme
  rating? This is the magnitude question proper: the sign is held fixed, only the size is left.
- `rho, angle only` against `rho, projection`: what the length adds to the angle.
- `rho, length vs extremity`: does the bare length of the vector, ignoring direction, track how far the rating
  is from the average? If yes, intensity is readable without knowing the trait: magnitude and direction are
  separate quantities. If length says nothing but the projection does, magnitude only exists along the direction.
- **Verdict:** *magnitude grades the trait* if the interval of the median `rho within a pole` is above zero,
  *sign only* if it is not (the state says which side, not how far).

persona-lsh's own answer on its seven traits was that sign bits identify the trait as well as any multi-bit
code, while the projection's size tracked the judge's score moderately (median rho 0.46).

In [ ]:
def eval_sets(X, raw, keyed):
    # Per factor: (centered eval vectors, their keyed ratings, their item indices, the train answers' mean rating).
    ok = np.isfinite(X).all(-1) & np.isfinite(raw)
    out = {}
    for g in FACTORS:
        centre = centres(X, ok, TRAIN[g])
        _, ty, _ = points(X, ok, keyed, TRAIN[g], centre)
        ex, ey, ei = points(X, ok, keyed, EVAL[g], centre)
        if len(ty) and len(ey):
            out[g] = (ex, ey, ei, ty.mean())
    return out


def separation(sets, dirs):
    # (cross[(f, g)] = rho of g's answers on f's direction, per-factor decomposition rows, summary numbers)
    cross = {(f, g): hx.spearman(ex @ dirs[f], ey) for g, (ex, ey, _, _) in sets.items() for f in dirs}
    leak = {}
    for g, (ex, ey, _, _) in sets.items():
        if g in dirs:
            own = float(np.cov(ex @ dirs[g], ey)[0, 1]) if len(ey) > 1 else 0.0
            for f in dirs:
                leak[f, g] = float(np.cov(ex @ dirs[f], ey)[0, 1]) / own if own > 0 else None
    rows = {}
    for g, (ex, ey, _, mid) in sets.items():
        if g not in dirs:
            continue
        p, length, high = ex @ dirs[g], np.linalg.norm(ex, axis=1), ey > mid
        inside = [r for r in (hx.spearman(p[high], ey[high]), hx.spearman(p[~high], ey[~high])) if r is not None]
        rows[g] = {"answers": len(ey), "rho, projection": cross[g, g],
                   "rho, angle only": hx.spearman(p / np.maximum(length, 1e-12), ey),
                   "pole accuracy": float(np.mean((p > 0) == high)), "share above average": float(high.mean()),
                   "rho within a pole": float(np.mean(inside)) if inside else None,
                   "rho, length vs extremity": hx.spearman(length, np.abs(ey - mid))}
    med = lambda v: float(np.median(v)) if len(v) else np.nan
    own = med([v for (f, g), v in cross.items() if f == g and v is not None])
    other = med([abs(v) for (f, g), v in cross.items() if f != g and v is not None])
    summary = {"own factor": own, "other factors": other,
               "leakage": med([abs(v) for (f, g), v in leak.items() if f != g and v is not None]),
               "within a pole": med([r["rho within a pole"] for r in rows.values() if r["rho within a pole"] is not None]),
               "pole accuracy": med([r["pole accuracy"] for r in rows.values()]),
               "length vs extremity": med([r["rho, length vs extremity"] for r in rows.values()
                                           if r["rho, length vs extremity"] is not None])}
    return cross, rows, summary, leak


def resample_items(sets, rng):
    # The same sets with each factor's eval items drawn with replacement (all of an item's answers together).
    out = {}
    for g, (ex, ey, ei, mid) in sets.items():
        items = np.unique(ei)
        take = np.concatenate([np.flatnonzero(ei == i) for i in rng.choice(items, len(items))])
        out[g] = (ex[take], ey[take], ei[take], mid)
    return out


SEP_BOOT = min(BOOT, 300)   # each resample recomputes the whole 6 x 6 table
CROSS, LEAK, rows, summaries = {}, {}, [], []
for name in RUN_NAMES:
    for kind in KINDS:
        X, dirs = slices(name, kind), DIRECTIONS.get((name, kind), {})
        if X is None or not dirs:
            continue
        sets = eval_sets(X, RAW[name], KEYED[name])
        cross, per, summary, leak = separation(sets, dirs)
        CROSS[name, kind] = pd.DataFrame([[cross.get((f, g)) for g in FACTORS] for f in FACTORS],
                                         index=FACTORS, columns=FACTORS, dtype=float)
        LEAK[name, kind] = pd.DataFrame([[leak.get((f, g)) for g in FACTORS] for f in FACTORS],
                                        index=FACTORS, columns=FACTORS, dtype=float)
        rows += [{"run": name, "kind": kind, "factor": g} | r for g, r in per.items()]
        rng = np.random.default_rng(BOOT_SEED)
        draws = pd.DataFrame([separation(resample_items(sets, rng), dirs)[2] for _ in range(SEP_BOOT)])
        ci = {k: tuple(np.nanpercentile(draws[k], [2.5, 97.5])) if draws[k].notna().any() else (np.nan, np.nan) for k in draws}
        between = ("no direction" if not ci["own factor"][0] > 0 else
                   "separable" if ci["leakage"][1] < 0.5 else "confused" if ci["leakage"][0] > 0.5 else "partly confused")
        magnitude = ("-" if between == "no direction" else
                     "magnitude grades the trait" if ci["within a pole"][0] > 0 else "sign only")
        summaries.append({"run": name, "kind": kind, **{k: summary[k] for k in ("own factor", "other factors", "leakage")},
                          "leakage 95%": f"[{ci['leakage'][0]:.2f}, {ci['leakage'][1]:.2f}]", "directions": between,
                          "pole accuracy": summary["pole accuracy"], "within a pole": summary["within a pole"],
                          "within 95%": f"[{ci['within a pole'][0]:+.2f}, {ci['within a pole'][1]:+.2f}]",
                          "length vs extremity": summary["length vs extremity"], "magnitude": magnitude})
SEPARATION = pd.DataFrame(summaries)
print("7a and 7b, per run and kind (medians over factors; intervals from resampling the eval items):\n")
show(SEPARATION, {"own factor": num, "other factors": two, "leakage": two, "pole accuracy": pct, "within a pole": num,
                  "length vs extremity": num})
SEPARATION.to_csv(RESULTS / "separation.csv", index=False)

DECOMPOSITION = pd.DataFrame(rows)
print("\n7b per factor:\n")
show(DECOMPOSITION, {c: num for c in DECOMPOSITION if c.startswith("rho")} | {"pole accuracy": pct, "share above average": pct})
DECOMPOSITION.to_csv(RESULTS / "direction_magnitude.csv", index=False)

def mean_table(tables):
    tables = [t for t in tables]
    return sum(t.fillna(0) for t in tables) / sum(t.notna().astype(int) for t in tables).replace(0, np.nan)


have = [k for k in CROSS if k[1] == GEO_KIND]
if have:
    mean_cross, mean_leak = mean_table(CROSS[k] for k in have), mean_table(LEAK[k] for k in have)
    cos = pd.DataFrame(index=FACTORS, columns=FACTORS, dtype=float)
    for a in FACTORS:
        for b in FACTORS:
            got = [float(d[a] @ d[b]) for (n, k), d in DIRECTIONS.items() if k == GEO_KIND and a in d and b in d]
            cos.loc[a, b] = np.mean(got) if got else np.nan
    fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.2))
    heat(axes[0], mean_leak, DIV, -1, 1, lambda v: f"{v:+.2f}")
    axes[0].set_title(f"7a. Leakage (mean over runs, {GEO_KIND})\nshare of a factor's signal landing on another's direction", fontsize=9)
    heat(axes[1], mean_cross, DIV, -1, 1, lambda v: f"{v:+.2f}")
    axes[1].set_title("rho with the ratings\ndiagonal = own factor", fontsize=9)
    for ax in axes[:2]:
        ax.set_xlabel("eval answers of factor")
    axes[0].set_ylabel("projected on the direction of factor")
    heat(axes[2], cos, DIV, -1, 1, lambda v: f"{v:+.2f}")
    axes[2].set_title("Cosine between the factor directions\nhuman scales: N against E and C, H with A", fontsize=9)
    fig.tight_layout()
    save(fig, "7_direction_separation")
    plt.show()
    mean_cross.to_csv(RESULTS / "cross_factor_rho.csv")
    mean_leak.to_csv(RESULTS / "cross_factor_leakage.csv")
    cos.to_csv(RESULTS / "factor_direction_cosines.csv")
    off = mean_leak.where(~np.eye(len(FACTORS), dtype=bool)).stack()
    if len(off):
        print("\nthe most confused pairs (mean over runs): direction -> other factor's answers")
        for (f, g), v in off.reindex(off.abs().sort_values(ascending=False).index).head(5).items():
            print(f"  {f} -> {g}: leakage {v:+.2f}, rho {mean_cross.loc[f, g]:+.2f}, "
                  f"cosine between the two directions {cos.loc[f, g]:+.2f}")

mine = DECOMPOSITION[DECOMPOSITION["kind"] == GEO_KIND] if len(DECOMPOSITION) else DECOMPOSITION
if len(mine):
    measures = [("rho, projection", "projection\n(angle and length)"), ("rho, angle only", "cosine\n(angle only)"),
                ("rho within a pole", "projection,\nwithin a pole"), ("rho, length vs extremity", "length vs\nextremity")]
    fig, ax = plt.subplots(figsize=(8.5, 3.4))
    for r, name in enumerate(RUN_NAMES):
        per = mine[mine["run"] == name]
        for m, (column, _) in enumerate(measures):
            v = pd.to_numeric(per[column], errors="coerce").dropna()
            offset = (r - (len(RUN_NAMES) - 1) / 2) * 0.8 / max(len(RUN_NAMES), 1)
            ax.scatter(np.full(len(v), m + offset), v, color=SLOTS[r % len(SLOTS)], marker=MARKERS[r % len(MARKERS)], s=16,
                       alpha=0.55, linewidth=0)
            if len(v):
                ax.hlines(v.median(), m + offset - 0.04, m + offset + 0.04, color=SLOTS[r % len(SLOTS)], linewidth=2.5)
    ax.axhline(0, color=BASELINE, linewidth=1)
    ax.set_xticks(range(len(measures)), [label for _, label in measures])
    ax.set_ylim(-1, 1)
    ax.set_ylabel("Spearman rho with the held-out ratings")
    ax.grid(axis="x", visible=False)
    ax.legend(handles=[Line2D([], [], color=SLOTS[r % len(SLOTS)], marker=MARKERS[r % len(MARKERS)], linestyle="", markersize=5,
                              label=name) for r, name in enumerate(RUN_NAMES)],
              loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8, title="run (dot = factor, bar = median)")
    ax.set_title(f"7b. What the angle, the length and the projection each say about the rating ({GEO_KIND})")
    fig.tight_layout()
    save(fig, "7_direction_magnitude")
    plt.show()

### 7c. Is a trait's direction the same under each prompt family?

The white-box side of section 3. The factor's direction is rebuilt from the train answers under one DAN level
only (its three prompts), and again per INFO level; the table gives the mean cosine between the levels'
directions. Near 1: the prompts move the model along one fixed direction (a stable trait axis, whatever the
score does). Near 0: each prompt family has its own direction for the "same" trait, so a library built under
one prompt reads the trait poorly under another. Blank where a level's answers don't vary.

In [ ]:
def level_directions(X, raw, keyed, f, groups):
    # One direction per group of conditions, from the factor's train answers under those conditions only.
    ok = np.isfinite(X).all(-1) & np.isfinite(raw)
    out = []
    for conds in groups:
        mask = np.zeros_like(ok)
        mask[conds] = ok[conds]
        d = hx.covariance_direction(*points(X, mask, keyed, TRAIN[f], centres(X, mask, TRAIN[f]))[:2])
        out.append(None if d is None or not np.linalg.norm(d) else d / np.linalg.norm(d))
    return out


def mean_cosine(dirs):
    got = [float(a @ b) for i, a in enumerate(dirs) for b in dirs[i + 1:] if a is not None and b is not None]
    return np.mean(got) if got else np.nan


DAN_GROUPS = [[ci for c, ci in ((c, CONDITIONS.index(c)) for c in CONDITIONS) if LEVELS[c][0] == d] for d in range(3)]
INFO_GROUPS = [[ci for c, ci in ((c, CONDITIONS.index(c)) for c in CONDITIONS) if LEVELS[c][1] == i] for i in range(3)]
rows = []
for name in RUN_NAMES:
    X = slices(name, GEO_KIND)
    if X is None:
        continue
    for f in FACTORS:
        rows.append({"run": name, "factor": f,
                     "across DAN levels": mean_cosine(level_directions(X, RAW[name], KEYED[name], f, DAN_GROUPS)),
                     "across INFO levels": mean_cosine(level_directions(X, RAW[name], KEYED[name], f, INFO_GROUPS))})
DIRECTION_STABILITY = pd.DataFrame(rows)
if len(DIRECTION_STABILITY):
    DIRECTION_STABILITY.to_csv(RESULTS / "direction_stability.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 0.42 * len(RUN_NAMES) + 1.5))
    for ax, column in zip(axes, ("across DAN levels", "across INFO levels")):
        table = DIRECTION_STABILITY.pivot(index="run", columns="factor", values=column).reindex(index=RUN_NAMES, columns=FACTORS)
        heat(ax, table, DIV, -1, 1, lambda v: f"{v:+.2f}")
        ax.set_title(f"Cosine between a factor's directions {column} ({GEO_KIND})", fontsize=9)
    axes[1].set_yticks([])
    fig.tight_layout()
    save(fig, "7c_direction_stability")
    plt.show()
    both = DIRECTION_STABILITY.merge(STABILITY[["run", "factor", "range", "prompts that move it"]], on=["run", "factor"])
    print("score stability (section 3) next to direction stability, per factor over runs (mean):\n")
    show(both.groupby("factor", sort=False)[["range", "prompts that move it", "across DAN levels", "across INFO levels"]]
         .mean().reset_index(), {"range": two, "prompts that move it": "{:.1f}".format, "across DAN levels": num,
                                 "across INFO levels": num})

## 8. The prompts as directions

persona-lsh's with minus without, with the differential prompt in place of the persona instruction: for each
item, `vector under the prompt - vector under the baseline`, one set of differences per non-baseline prompt
(eight classes, chance 1/8). The class means come from the train items; classification is cross-validated on
the train items and held-out on the eval items, by cosine and by RP-LSH digest.

`DAN right` and `INFO right` say which axis the held-out cosine picks got right, so the misses show whether
it is the DAN phrase or the INFO phrase that the vector loses.

A caution carried over from persona-lsh's "instruction or behavior" question: the prompts differ in their
words, and the `prompt` vector has read those words. High accuracy says a digest identifies which differential
prompt was used, which is the monitoring claim. It doesn't yet say the model behaves differently: that is
sections 3 and 4, and section 9 joins the two.

The second table is each prompt's lean: the cosine between its mean difference and each factor's direction.

In [ ]:
def prompt_sets(X):
    # (train, test) dicts by non-baseline prompt: differences from the baseline, train items and eval items.
    diffs = {c: X[CONDITIONS.index(c)] - X[BASE] for c in OTHERS}
    return {c: d[ALL_TRAIN] for c, d in diffs.items()}, {c: d[ALL_EVAL] for c, d in diffs.items()}


rows, LEAN = [], []
for name in RUN_NAMES:
    for kind in KINDS:
        X = slices(name, kind)
        if X is None:
            continue
        train, test = prompt_sets(X)
        held = classify(train, test)
        row = {"run": name, "kind": kind, "chance": 1 / len(OTHERS)} | acc(cross_validate(train), "cv ") | acc(held, "held-out ")
        for axis, k in (("DAN", 0), ("INFO", 1)):
            row[f"{axis} right"] = np.mean([hx.levels(r["cosine"])[k] == hx.levels(r["true"])[k] for r in held]) if held else np.nan
        rows.append(row)
        for c in OTHERS:   # which way each prompt leans: its mean difference against each factor direction
            mean = finite(train[c]).mean(0)
            LEAN.append({"run": name, "kind": kind, "prompt": c} | {
                f: float(mean @ d / max(np.linalg.norm(mean), 1e-12)) for f, d in DIRECTIONS[name, kind].items()})
PROMPT_ID = pd.DataFrame(rows)
show(PROMPT_ID, {c: pct for c in PROMPT_ID if c not in ("run", "kind") and not c.endswith(" n")})
PROMPT_ID.to_csv(RESULTS / "prompt_identification.csv", index=False)

LEAN = pd.DataFrame(LEAN)
if len(LEAN):
    print(f"\ncosine between each prompt's direction and each factor direction (kind {GEO_KIND}, mean over runs):\n")
    lean = LEAN[LEAN["kind"] == GEO_KIND].groupby("prompt", sort=False)[[f for f in FACTORS if f in LEAN]].mean()
    show(lean, {f: num for f in FACTORS}, index=True)
    LEAN.to_csv(RESULTS / "prompt_lean.csv", index=False)

## 9. The join: does the white-box shift match the black-box shift?

PLAN section 5, part C. For each non-baseline prompt and each factor, on the **eval** items rated under both
that prompt and the baseline:
- the **score shift**: mean keyed rating under the prompt minus under the baseline (section 4, eval half only);
- the **projection shift**: the mean of (vector under the prompt minus vector under the baseline), projected on
  the factor's direction from section 6 (train half only).

Per run that is up to 8 prompts x 6 factors = 48 points. `rho` is their Spearman correlation, `same sign` the
share of points with a non-zero score shift where the two shifts agree in sign (chance 50%). If they match, the
digest is a cheaper, pre-output readout of the personality shift an outside test measures. Few points and
small shifts make this the least certain table here: read it across runs, not run by run.

In [ ]:
rows, POINTS = [], []
for name in RUN_NAMES:
    for kind in KINDS:
        X = slices(name, kind)
        if X is None:
            continue
        pts = []
        for c in OTHERS:
            ci = CONDITIONS.index(c)
            for f, d in DIRECTIONS[name, kind].items():
                diff = KEYED[name][ci][EVAL[f]] - KEYED[name][BASE][EVAL[f]]
                move = (X[ci][EVAL[f]] - X[BASE][EVAL[f]]) @ d
                use = np.isfinite(diff) & np.isfinite(move)
                if use.any():
                    pts.append({"run": name, "kind": kind, "prompt": c, "factor": f, "items": int(use.sum()),
                                "score shift": diff[use].mean(), "projection shift": move[use].mean()})
        POINTS += pts
        s, p = np.array([q["score shift"] for q in pts]), np.array([q["projection shift"] for q in pts])
        moved = s != 0
        rows.append({"run": name, "kind": kind, "points": len(pts), "rho": hx.spearman(p, s) if len(pts) else None,
                     "with a score shift": int(moved.sum()),
                     "same sign": np.mean(np.sign(s[moved]) == np.sign(p[moved])) if moved.any() else np.nan})
JOIN = pd.DataFrame(rows)
show(JOIN, {"rho": num, "same sign": pct})
JOIN.to_csv(RESULTS / "join.csv", index=False)
POINTS = pd.DataFrame(POINTS)
POINTS.to_csv(RESULTS / "join_points.csv", index=False)
if len(POINTS):
    print("\nper factor, all runs pooled:\n")
    per = [{"kind": k, "factor": f, "points": len(g), "rho": hx.spearman(g["projection shift"], g["score shift"])}
           for (k, f), g in POINTS.groupby(["kind", "factor"], sort=False)]
    show(pd.DataFrame(per), {"rho": num})

In [ ]:
mine = POINTS[POINTS["kind"] == GEO_KIND] if len(POINTS) else POINTS
if len(mine):
    names = [n for n in RUN_NAMES if (mine["run"] == n).any()]
    fig, axes = grid_of(len(names), per_row=4, width=3.1, height=2.9)
    for ax, name in zip(axes, names):
        per = mine[mine["run"] == name]
        scale = per["projection shift"].abs().max() or 1.0
        ax.axhline(0, color=BASELINE, linewidth=1)
        ax.axvline(0, color=BASELINE, linewidth=1)
        ax.scatter(per["score shift"], per["projection shift"] / scale, color=BLUE, s=22, alpha=0.75,
                   edgecolor=SURFACE, linewidth=0.8)
        rho = hx.spearman(per["projection shift"], per["score shift"])
        ax.set_title(f"{name}   rho {'n/a' if rho is None else f'{rho:+.2f}'}", fontsize=9)
        ax.set_xlabel("score shift (points)")
    axes[0].set_ylabel("projection shift (scaled to its largest)")
    fig.suptitle(f"One dot per prompt and factor: shift along the factor's direction against shift in its score ({GEO_KIND})",
                 x=0.01, ha="left", fontsize=11, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    save(fig, "9_join")
    plt.show()

## 10. Layer sweep

Sections 5, 6 and 8 at every layer, cosine only: held-out factor identification, the median over factors of
the personality direction's rho on the eval half, and held-out prompt identification. Depth is layer / layers,
so models of different depth line up. persona-lsh's persona signal peaked at 59-71% of depth.

In [ ]:
rows, CURVES = [], {}
for name in RUN_NAMES:
    for kind in KINDS:
        S = full(name, kind)
        if S is None:
            continue
        curve = []
        for layer in range(1, S.shape[2]):
            X = S[:, :, layer].astype(np.float32)
            rhos = [r.get("rho, factor direction") for r in personality_directions(X, RAW[name], KEYED[name])[1]]
            rhos = [r for r in rhos if r is not None]
            curve.append({"layer": layer, "depth": layer / N_LAYERS[name],
                          "factor id": acc(classify(*factor_sets(X), rp=False))["cosine"],
                          "personality rho": float(np.median(rhos)) if rhos else np.nan,
                          "prompt id": acc(classify(*prompt_sets(X), rp=False))["cosine"]})
        CURVES[f"{name} {kind}"] = curve
        row = {"run": name, "kind": kind, "layers": N_LAYERS[name], "default layer": default_layer(N_LAYERS[name])}
        for key in ("factor id", "personality rho", "prompt id"):
            vals = [c[key] for c in curve]
            if np.isfinite(vals).any():
                best = curve[int(np.nanargmax(vals))]
                row |= {f"{key}: best": best[key], f"{key}: at depth": best["depth"],
                        f"{key}: at default": curve[row["default layer"] - 1][key]}
        rows.append(row)
        del S
SWEEP = pd.DataFrame(rows)
show(SWEEP, {c: (num if "rho" in c and "depth" not in c else pct) for c in SWEEP if ":" in c})
SWEEP.to_csv(RESULTS / "layer_sweep.csv", index=False)
_ = (RESULTS / "layer_curves.json").write_text(json.dumps(CURVES, indent=1))

In [ ]:
panels = [("factor id", "5. Factor identification (held-out)", True), ("personality rho", "6. Personality direction (median rho)", False),
          ("prompt id", "8. Prompt identification (held-out)", True)]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, (key, title, share) in zip(axes, panels):
    for r, name in enumerate(RUN_NAMES):
        curve = CURVES.get(f"{name} {GEO_KIND}")
        if curve:
            ax.plot([c["depth"] for c in curve], [c[key] for c in curve], color=SLOTS[r % len(SLOTS)],
                    marker=MARKERS[r % len(MARKERS)], markersize=3.5, linewidth=1.6, label=name)
    ax.axvline(20 / 28, color=MUTED, linewidth=1, linestyle=(0, (3, 2)))
    ax.set_xlabel("depth (layer / layers)")
    ax.set_title(title, fontsize=9)
    if share:
        ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    else:
        ax.axhline(0, color=BASELINE, linewidth=1)
axes[0].text(20 / 28, 0.02, " default layer", transform=axes[0].get_xaxis_transform(), fontsize=8, color=MUTED)
axes[2].legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)
fig.suptitle(f"By depth, {GEO_KIND} vectors", x=0.01, ha="left", fontsize=11, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.94))
save(fig, "10_layer_sweep")
plt.show()

## 11. The cards during the run

From the run's telemetry directory (`~/gpu-telemetry/persona-hexaco_*`): `events.csv` (each step's start and
end) and `gpus.csv` (per card, once a second). Time per run is split into load, generation, hidden-state read
and judging. Power is the sum over the model cards per sample. Kernel lines (`kernel.log`) are placed at the
step that was running: a card drop (Xid 79) shows here with the model and prompt it happened in.

In [ ]:
def has_capture(d):
    # A rerun that only redid the ratings has no " gen" steps; its telemetry says nothing about capture.
    try:
        return pd.read_csv(d / "events.csv")["step"].astype(str).str.endswith(" gen").any()
    except (OSError, KeyError, pd.errors.ParserError):
        return False


tdir = (TELEMETRY_DIR if TELEMETRY_DIR and has_capture(Path(TELEMETRY_DIR)) else
        max((d for d in TELEMETRY_ROOT.glob("persona-hexaco_*") if has_capture(d)), default=None, key=lambda p: p.name))
if not tdir or not (Path(tdir) / "events.csv").exists():
    print("No persona-hexaco telemetry directory with a capture under", TELEMETRY_ROOT)
else:
    tdir = Path(tdir)
    print("telemetry:", tdir)
    ev = pd.read_csv(tdir / "events.csv")
    cards = [int(c) for c in str(ev["card"].iloc[0]).split("+")]
    windows, starts = [], {}
    for e in ev.itertuples():
        if e.event == "start":
            starts[e.step] = e.epoch
        elif e.event in ("end", "error") and e.step in starts:
            windows.append({"step": e.step, "t0": starts.pop(e.step), "t1": e.epoch, "event": e.event})
    for step_name, t0 in starts.items():   # a step that never ended: the run died in it
        windows.append({"step": step_name, "t0": t0, "t1": ev["epoch"].max(), "event": "unfinished"})
    W = pd.DataFrame(windows)
    W["seconds"] = W["t1"] - W["t0"]
    run_names = [p.name for p in OUT_DIR.iterdir() if p.is_dir() and p.name != "results"]   # unfinished runs too

    def phase_of(step_name):
        for run in sorted(run_names, key=len, reverse=True):   # longest first: "qwen3-8b-think" before "qwen3-8b"
            if step_name.startswith(f"judge {run} "):
                return run, "judge"
            if step_name.startswith(f"{run} "):
                return run, ("read" if step_name.endswith(" read") else "load" if step_name.endswith(" load") else "generate")
        return None, step_name.split()[0]

    W[["run", "phase"]] = W["step"].apply(lambda s: pd.Series(phase_of(s)))
    phases = ["load", "generate", "read", "judge"]
    hours = W.dropna(subset=["run"]).pivot_table(index="run", columns="phase", values="seconds", aggfunc="sum",
                                                 fill_value=0).reindex(columns=phases, fill_value=0) / 3600
    order = list(dict.fromkeys(W.dropna(subset=["run"])["run"]))   # in the order they ran
    hours = hours.reindex(order)

    g = pd.read_csv(tdir / "gpus.csv", skipinitialspace=True, on_bad_lines="skip")
    g = g[pd.to_numeric(g["index"], errors="coerce").notna()].astype({"index": int})
    g["power.draw"] = pd.to_numeric(g["power.draw"], errors="coerce")
    g["temperature.gpu"] = pd.to_numeric(g["temperature.gpu"], errors="coerce")
    mine = g[g["index"].isin(cards)]
    total = mine.groupby("epoch")["power.draw"].agg(["sum", "count"])
    total = total[total["count"] == len(cards)]["sum"]

    def in_window(series, t0, t1):
        return series[(series.index >= t0) & (series.index <= t1)]

    rows = []
    for run in hours.index:
        w = W[W["run"] == run]
        capture = w[w["phase"] != "judge"]
        cap = pd.concat([in_window(total, r.t0, r.t1) for r in capture.itertuples()]) if len(capture) else pd.Series(dtype=float)
        temps = mine[mine["epoch"].between(capture["t0"].min(), capture["t1"].max())] if len(capture) else mine.iloc[:0]
        rows.append({"run": run, **{f"{p} h": hours.loc[run, p] for p in phases},
                     "capture mean W": cap.mean(), "capture max W": cap.max(),
                     **{f"card {c} max C": temps[temps["index"] == c]["temperature.gpu"].max() for c in cards},
                     "unfinished steps": "; ".join(w[w["event"] != "end"]["step"]) or "-"})
    TEL = pd.DataFrame(rows)
    print(TEL.round(2).to_string(index=False))
    TEL.to_csv(RESULTS / "telemetry.csv", index=False)
    print(f"\nwhole run: {(ev['epoch'].max() - ev['epoch'].min()) / 3600:.1f} h; "
          f"total W across cards {'+'.join(map(str, cards))}: mean {total.mean():.0f}, max {total.max():.0f}")

    gen = W[W["phase"] == "generate"].copy()
    if len(gen):   # which prompt costs the most: generation seconds per condition, summed over runs
        gen["prompt"] = gen["step"].str.split().str[-2]
        print("\ngeneration minutes per prompt, per run:")
        print((gen.pivot_table(index="run", columns="prompt", values="seconds", aggfunc="sum") / 60)
              .reindex(index=order, columns=CONDITIONS).round(1).to_string(na_rep=""))

    fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), gridspec_kw={"height_ratios": [1, 1.1]})
    ax = axes[0]
    y = np.arange(len(hours))[::-1]
    left = np.zeros(len(hours))
    for i, p in enumerate(phases):
        ax.barh(y, hours[p], left=left, color=SLOTS[i], height=0.6, edgecolor=SURFACE, linewidth=1, label=p)
        left += hours[p].to_numpy()
    for yi, h in zip(y, left):
        ax.text(h, yi, f"  {h:.1f} h" if h >= 0.1 else f"  {h * 60:.0f} min", va="center", fontsize=8, color=INK2)
    ax.set_yticks(y, hours.index)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("hours")
    ax.set_xlim(0, max(left.max(), 1e-9) * 1.15)
    ax.legend(loc="lower right", ncol=4, bbox_to_anchor=(1, 1.0))
    ax.set_title("Wall time per run")

    ax = axes[1]
    start = ev["epoch"].min()
    smooth = total.rolling(60, min_periods=1).mean()
    ax.plot((total.index - start) / 3600, total.to_numpy(), color=BLUE, linewidth=0.4, alpha=0.35)
    ax.plot((smooth.index - start) / 3600, smooth.to_numpy(), color=BLUE, linewidth=1.5,
            label=f"cards {'+'.join(map(str, cards))}, 60 s mean")
    for i, run in enumerate(order):
        capture = W[(W["run"] == run) & (W["phase"] != "judge")]
        if len(capture):
            a, b = (capture["t0"].min() - start) / 3600, (capture["t1"].max() - start) / 3600
            ax.axvspan(a, b, color=GRID, alpha=0.6 if i % 2 else 0.25, linewidth=0)
            ax.text((a + b) / 2, 1.0, run, transform=ax.get_xaxis_transform(), ha="center", va="bottom",
                    fontsize=7, color=INK2, rotation=30)
    klog = tdir / "kernel.log"
    KERNEL = []
    for line in (klog.read_text().splitlines() if klog.exists() else []):
        try:
            t = float(line.split()[0])
        except (ValueError, IndexError):
            continue
        hit = W[(W["t0"] <= t) & (W["t1"] >= t)]
        KERNEL.append({"hours in": (t - start) / 3600, "step": "; ".join(hit["step"]) or "(between steps)",
                       "line": " ".join(line.split()[2:])[:110]})
    for i, t in enumerate(sorted({round(k["hours in"], 3) for k in KERNEL})):
        ax.axvline(t, color=SERIOUS, linewidth=1, linestyle=(0, (2, 2)),
                   label="kernel message (AER/NVRM)" if i == 0 else None)
    ax.set_xlabel("hours since the notebook started")
    ax.set_ylabel("watts")
    ax.legend(loc="lower left", fontsize=8)
    ax.set_title("Total power on the model cards (shaded: each run's capture)", pad=34)
    fig.tight_layout()
    save(fig, "11_telemetry")
    plt.show()

    print("\nHighest one-second samples per card, and the step running:")
    for c in cards:
        for _, r in mine[mine["index"] == c].nlargest(3, "power.draw").iterrows():
            hit = W[(W["t0"] <= r["epoch"]) & (W["t1"] >= r["epoch"])]
            print(f"  card {c}: {r['power.draw']:.0f} W (limit {r['power.limit']}) at "
                  f"{(r['epoch'] - start) / 3600:.2f} h, {'; '.join(hit['step']) or '(between steps)'}")

    if KERNEL:
        print(f"{len(KERNEL)} kernel lines:")
        print(pd.DataFrame(KERNEL).round(2).to_string(index=False))
        buses = g.drop_duplicates("index").set_index("pci.bus_id")["index"].to_dict()
        print("bus -> card:", {b[-7:]: c for b, c in buses.items()})
    else:
        print("kernel messages: none")

## 12. What to read off, once the first experiment is in

Nothing below is a finding yet: the notebook has only run on synthetic and stand-in data. Each line names the
table or figure that answers it.

1. **What does each model's profile look like, and how even is it?** The polar diagrams: the shape, the width
   of the band, and how far each of the nine departs from the dashed baseline (1).
2. **Do the models answer at all, and what does it cost?** Rated share and reply length per prompt, whether DAN2
   runs into the token limit, minutes per run (2, 11).
3. **Which traits are stable and which aren't?** `prompts that move it` and `range` per model and factor; whether
   the same factors are unstable across models; and whether an unstable score comes with an unstable direction
   (3, 7c).
4. **What do the prompts do overall?** The main-effects chart. Does INFO0 raise N and C? Does DAN move C and A?
   Are small models bland? Do DAN and INFO add up? (4)
5. **Do digests keep the factor signal?** RP-LSH at 256 and 1024 bits against cosine, held-out, and under the
   other prompts (5).
6. **Is there a personality direction beyond agreement?** `rho, factor direction` against `rho, agreement
   direction`, and `rho, same item across prompts`, on the `prompt` vectors (6).
7. **Are direction and magnitude separable or confused?** The `directions` and `magnitude` verdicts per run,
   the cross-factor heatmap for which pairs are confused, and whether those pairs are the ones the human scales
   relate (7a, 7b).
8. **Can a digest tell which differential prompt was used, and which axis is easier?** (8)
9. **Does the white-box shift track the black-box shift?** (9)
10. **Is the best layer where persona-lsh's was?** (10)

Limits to print with any result: one seed (one train/eval cut); 12 train items per Big Five factor and 8 for
H; one judge, used only for replies that aren't a bare number; greedy decoding, so no within-prompt variance
(the band in section 1 is spread over items, not over repeated answers); the intervals resample items within
the models run, they don't generalize to other models; the nine prompts differ in wording, so prompt
identification partly reads words; the HEXACO items are used under hexaco.org's non-profit academic terms.